# Multi-Face Video FaceSwap — one-click Colab

Runs the full pipeline (InsightFace `buffalo_l` + `inswapper_128`) and prints a public Gradio URL.

**Steps:**
1. *(optional)* Runtime ▸ Change runtime type ▸ pick a **T4 GPU**.
2. `Runtime ▸ Run all`.
3. Open the `https://*.gradio.live` URL the last cell prints.

In [ ]:
# 1. Clone the repository (default branch already has all the code)
!git clone --depth 1 https://github.com/royaleagleweb/FACESWAP.git
%cd FACESWAP

In [ ]:
# 2. Install only the deps the core engine needs.
#    Skip gfpgan/basicsr — they pin against an old torchvision and break
#    on Colab's image. The UI works fine without face enhancement.
!apt-get -qq update && apt-get -qq install -y ffmpeg
!pip install -q \
    "numpy<2.1" \
    "opencv-python-headless>=4.8" \
    "onnx>=1.14" \
    "onnxruntime-gpu" \
    "insightface==0.7.3" \
    "ffmpeg-python" \
    "tqdm" \
    "pillow" \
    "gradio>=4.40" \
    "requests" \
    "filetype"

In [ ]:
# 3. Pre-stage models so the first request is instant
import os, urllib.request, zipfile, pathlib
pathlib.Path('models').mkdir(exist_ok=True)
pathlib.Path('/root/.insightface/models').mkdir(parents=True, exist_ok=True)

if not os.path.exists('models/inswapper_128.onnx'):
    print('Downloading inswapper_128.onnx (~530 MB)...')
    urllib.request.urlretrieve(
        'https://github.com/facefusion/facefusion-assets/releases/download/models-3.0.0/inswapper_128.onnx',
        'models/inswapper_128.onnx',
    )

if not os.path.exists('/root/.insightface/models/buffalo_l/det_10g.onnx'):
    print('Downloading buffalo_l (~290 MB)...')
    urllib.request.urlretrieve(
        'https://github.com/deepinsight/insightface/releases/download/v0.7/buffalo_l.zip',
        '/root/.insightface/models/buffalo_l.zip',
    )
    with zipfile.ZipFile('/root/.insightface/models/buffalo_l.zip') as z:
        z.extractall('/root/.insightface/models/buffalo_l')

print('Models ready.')

In [ ]:
# 4. Launch the Gradio UI with a public share link
from ui.app import build_app
app = build_app()
app.queue().launch(share=True)